# Chapitre 15 - Exercices 15.1 à 15.4

- **15.1** : variance d'une moyenne de variables équicorrélées, et le cas $\rho<0$ ;
- **15.2** : erreur *out-of-bag* et validation croisée « leave-one-out » ;
- **15.3** : le modèle de simulation de Mease et Wyner (figure 15.7) et son erreur de Bayes ;
- **15.4** : corrélation entre deux moyennes bootstrap, et inefficacité du bagging pour
  une statistique linéaire.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch15_forets as rf
set_style()

## Exercice 15.1 : Variance d'une moyenne de variables corrélées

**Énoncé.** Établir la formule (15.1) : si $T_1,\dots,T_B$ ont toutes la variance
$\sigma^2$ et une corrélation deux à deux égale à $\rho$, alors
$$\operatorname{Var}\Big(\frac1B\sum_{b=1}^BT_b\Big)=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2.$$
Cette formule semble mise en défaut lorsque $\rho<0$ ; expliquer le problème.

### Rappels

- Covariance : $\operatorname{Cov}(U,V)=\mathbb E[(U-\mathbb EU)(V-\mathbb EV)]$ ; elle est
  **bilinéaire** et symétrique, et $\operatorname{Cov}(U,U)=\operatorname{Var}U$.
- Corrélation : $\operatorname{corr}(U,V)=\operatorname{Cov}(U,V)/\sqrt{\operatorname{Var}U\operatorname{Var}V}$,
  donc ici $\operatorname{Cov}(T_b,T_{b'})=\rho\sigma^2$ pour $b\neq b'$.
- Pour un vecteur aléatoire $T=(T_1,\dots,T_B)^T$ de matrice de covariance $\Sigma$ et un
  vecteur fixe $a$, $\operatorname{Var}(a^TT)=a^T\Sigma a\ge0$ : une matrice de covariance
  est toujours **symétrique positive** (semi-définie positive).

### Démonstration de (15.1)

Par bilinéarité de la covariance,
$$\operatorname{Var}\Big(\sum_bT_b\Big)=\operatorname{Cov}\Big(\sum_bT_b,\sum_{b'}T_{b'}\Big)=\sum_{b}\sum_{b'}\operatorname{Cov}(T_b,T_{b'}).$$
La double somme comporte $B$ termes diagonaux ($b=b'$), chacun égal à $\sigma^2$, et
$B(B-1)$ termes hors diagonale, chacun égal à $\rho\sigma^2$. Donc
$$\operatorname{Var}\Big(\sum_bT_b\Big)=B\sigma^2+B(B-1)\rho\sigma^2 .$$
Comme $\operatorname{Var}(cU)=c^2\operatorname{Var}U$, on divise par $B^2$ :
$$\operatorname{Var}\Big(\frac1B\sum_bT_b\Big)=\frac{\sigma^2}{B}+\frac{B-1}{B}\rho\sigma^2
=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2=\frac{\sigma^2}{B}\big(1+(B-1)\rho\big).\qquad\blacksquare$$

### Le cas $\rho<0$

Si $\rho<0$ est fixé, le terme $\rho\sigma^2$ est négatif et la formule donne une
variance **négative** dès que $1+(B-1)\rho<0$, c'est-à-dire dès que $B>1-1/\rho$. Une
variance ne pouvant être négative, quelque chose ne va pas. Le diagnostic : **l'hypothèse
elle-même est impossible**. On ne peut pas avoir $B$ variables deux à deux corrélées
négativement avec la même corrélation $\rho$ quand $B$ est grand.

Pour le voir, écrivons la matrice de covariance
$\Sigma=\sigma^2\big[(1-\rho)\mathbf I+\rho\,\mathbf 1\mathbf 1^T\big]$, où $\mathbf 1$ est
le vecteur de $B$ uns. Cherchons ses valeurs propres :

- $\Sigma\mathbf 1=\sigma^2\big[(1-\rho)\mathbf 1+\rho B\mathbf 1\big]=\sigma^2(1+(B-1)\rho)\,\mathbf 1$
  (car $\mathbf 1^T\mathbf 1=B$) : valeur propre $\lambda_1=\sigma^2(1+(B-1)\rho)$ ;
- pour tout $v\perp\mathbf 1$ (sous-espace de dimension $B-1$), $\mathbf 1^Tv=0$ donc
  $\Sigma v=\sigma^2(1-\rho)v$ : valeur propre $\sigma^2(1-\rho)$, de multiplicité $B-1$.

$\Sigma$ est positive si et seulement si ses valeurs propres sont $\ge0$, soit
$$-\frac1{B-1}\le\rho\le1 .$$

Remarquons que la variance de la moyenne vaut exactement
$\operatorname{Var}(\bar T)=\frac1{B^2}\mathbf 1^T\Sigma\mathbf 1=\lambda_1/B$ : elle est
positive **précisément** quand la matrice de covariance est admissible. La formule ne
« échoue » donc jamais pour une configuration qui existe vraiment. Pour $B$ variables, la
corrélation commune ne peut pas descendre en dessous de $-1/(B-1)$, qui tend vers 0 quand
$B\to\infty$ : des variables nombreuses ne peuvent pas être toutes négativement corrélées
entre elles de façon notable.

Vérifions numériquement : pour $\rho=-0{,}1$, la plus petite valeur propre de $\Sigma$
et la formule (15.1) changent de signe au même endroit, $B=1-1/\rho=11$.

In [ ]:
rho = -0.1
B_grille = np.arange(2, 21)
vp_min = [np.linalg.eigvalsh((1 - rho) * np.eye(B) + rho * np.ones((B, B))).min() for B in B_grille]
formule = rf.variance_moyenne_correlee(1.0, rho, B_grille)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(B_grille, formule, "o-", color=BLUE, label=r"formule (15.1) : $\sigma^2(1+(B-1)\rho)/B$")
ax.plot(B_grille, np.array(vp_min) / B_grille, "s--", color=ORANGE, label=r"(plus petite valeur propre de $\Sigma$)$/B$")
ax.axhline(0, color="k", lw=0.8)
ax.axvline(1 - 1 / rho, color=GRAY, ls=":", label=r"$B=1-1/\rho=11$")
ax.set_xlabel("$B$")
ax.set_ylabel(r"valeur ($\sigma^2=1$)")
ax.set_title(r"Exercice 15.1 : $\rho=-0{,}1$, la covariance n'existe plus pour $B>11$")
ax.legend()
plt.show()

Au-delà de $B=11$, la plus petite valeur propre (et donc la formule, qui en est un
multiple pour $\mathbf 1$ dès que $\rho<0$) est négative : $\Sigma$ n'est pas une
matrice de covariance.

Le cas limite $\rho=-1/(B-1)$ est atteint : si $E_1,\dots,E_B$ sont i.i.d.
$\mathcal N(0,1)$ et $T_b=E_b-\bar E$, les $T_b$ ont la même variance, une corrélation
commune $-1/(B-1)$, et leur moyenne est identiquement nulle (variance 0, conforme à la
formule).

In [ ]:
rng = np.random.RandomState(0)
B = 6
E = rng.randn(100000, B)
T = E - E.mean(axis=1, keepdims=True)
C = np.corrcoef(T.T)
print("corrélation moyenne hors diagonale : %.4f   (-1/(B-1) = %.4f)" % (C[~np.eye(B, dtype=bool)].mean(), -1 / (B - 1)))
print("variance de la moyenne : %.6f   formule : %.6f" % (T.mean(axis=1).var(), abs(rf.variance_moyenne_correlee(T.var(0).mean(), -1 / (B - 1), B))))

> **À retenir.** $\operatorname{Var}(\bar T)=\rho\sigma^2+(1-\rho)\sigma^2/B$ s'obtient par
> bilinéarité de la covariance. Une corrélation commune négative n'est possible que si
> $\rho\ge-1/(B-1)$ ; dans ce domaine la formule reste positive. Pour les forêts, $\rho\ge0$
> en pratique et le plancher $\rho\sigma^2$ est la quantité à réduire.

## Exercice 15.2 : Erreur OOB et validation croisée à $N$ blocs

**Énoncé.** Montrer que lorsque le nombre $B$ d'échantillons bootstrap devient grand,
l'estimation OOB de l'erreur d'une forêt aléatoire tend vers son estimation par
validation croisée à $N$ blocs (*leave-one-out*), et que l'identité est exacte à la limite.

### Notations

Un échantillon bootstrap est une suite d'indices $I_1,\dots,I_N$ **i.i.d. uniformes** sur
$\{1,\dots,N\}$ ; l'arbre correspondant dépend de ces indices et d'un aléa supplémentaire
(tirage des $m$ variables), indépendant. Notons $\mathbf Z_{-i}$ l'échantillon privé de
$z_i$.

- **Prédiction OOB de $z_i$** : moyenne (ou vote) des arbres dont l'échantillon bootstrap
  ne contient pas $i$.
- **Validation croisée à $N$ blocs** : pour chaque $i$, on ajuste une forêt sur
  $\mathbf Z_{-i}$ et on prédit $y_i$ ; l'erreur CV est la moyenne des erreurs commises.

### Étape 1 : un échantillon bootstrap qui évite $i$ est un échantillon bootstrap de $\mathbf Z_{-i}$

Conditionnons par l'événement $A_i=\{I_k\neq i\ \text{pour tout }k\}$. Comme les $I_k$
sont indépendants et que $A_i=\bigcap_k\{I_k\ne i\}$, ils restent **indépendants**
conditionnellement à $A_i$, et chacun est uniforme sur $\{1,\dots,N\}\setminus\{i\}$ :
pour $j\neq i$,
$$\mathbb P(I_k=j\mid I_k\neq i)=\frac{1/N}{(N-1)/N}=\frac1{N-1}.$$
Autrement dit, conditionnellement à $A_i$, l'échantillon est **exactement** un échantillon
bootstrap de taille $N$ tiré dans $\mathbf Z_{-i}$, et l'arbre construit a la loi d'un
arbre de forêt ajusté sur $\mathbf Z_{-i}$.

### Étape 2 : loi des grands nombres

Les arbres $b=1,\dots,B$ sont i.i.d. (conditionnellement aux données). Le nombre $B_i$
d'arbres pour lesquels $z_i$ est OOB suit une loi binomiale
$\mathcal B\big(B,(1-1/N)^N\big)$, donc $B_i\to\infty$ presque sûrement quand $B\to\infty$.
Ces $B_i$ arbres sont i.i.d. de la loi décrite à l'étape 1. Par la loi des grands nombres,
$$\hat f^{\rm oob}(x_i)=\frac1{B_i}\sum_{b:\,i\notin\mathbf Z^{*b}}T_b(x_i)\ \xrightarrow[B\to\infty]{}\ \mathbb E_{\Theta}\big[T(x_i;\Theta(\mathbf Z_{-i}))\big]=\hat f_{\rm rf}^{(-i)}(x_i),$$
la forêt « limite » (15.3) ajustée sur $\mathbf Z_{-i}$. C'est aussi la limite, quand son
nombre d'arbres tend vers l'infini, de la forêt ajustée sur $\mathbf Z_{-i}$ dans la
validation croisée.

### Conclusion

Pour chaque $i$, la prédiction OOB et la prédiction CV leave-one-out ont la **même
limite** ; l'erreur étant une moyenne finie ($N$ termes) de fonctions continues de ces
prédictions (pour l'erreur quadratique ; pour le vote majoritaire, hors cas d'égalité
exacte à $1/2$), les deux estimations d'erreur ont la même limite : l'identité est exacte
quand $B\to\infty$.

> **Remarque.** Pour une égalité exacte, la forêt de validation croisée doit tirer des
> échantillons bootstrap de taille $N$ dans les $N-1$ observations de $\mathbf Z_{-i}$ (c'est
> ce que « voit » un arbre OOB). Avec la convention usuelle (taille $N-1$), la différence
> est minime.

### Vérification numérique

Petit problème de régression ($N=30$, $p=3$), arbres de forêt avec $m=2$ et $n_{\min}=3$.
Pour la validation croisée, nous ajustons pour chaque $i$ 1000 arbres sur des échantillons
bootstrap de taille $N$ tirés dans $\mathbf Z_{-i}$ ; pour l'OOB, une forêt de $B$ arbres
avec $B$ croissant.

In [ ]:
rng = np.random.RandomState(0)
N, p_ex = 30, 3
X = rng.rand(N, p_ex)
y = np.sin(4 * X[:, 0]) + X[:, 1] + 0.3 * rng.randn(N)

t0 = time.time()
B_cv = 1000
pred_cv = np.zeros(N)
for i in range(N):
    autres = np.delete(np.arange(N), i)
    s = 0.0
    for b in range(B_cv):
        boot = rng.choice(autres, N)
        s += rf.ArbreAleatoire(m=2, nmin=3, rng=rng).fit(X[boot], y[boot]).predict(X[i:i + 1])[0]
    pred_cv[i] = s / B_cv
err_cv = np.mean((pred_cv - y) ** 2)
print("erreur CV leave-one-out : %.4f   (%.0f s)" % (err_cv, time.time() - t0))